In [1]:
!pip install -qU \
    "langchain-docling[local]" \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    langchain-chroma \
    chromadb \
    langchain-groq \
    pymupdf \
    pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.1.1 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
holidays 

In [2]:
import os
import glob
import re
import json
import hashlib
import fitz
import pandas as pd

from langchain_docling import DoclingLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [3]:
PDF_FOLDER = "/kaggle/input/datasets/priyankaadheli1/dataset"

pdf_files = glob.glob(
    os.path.join(PDF_FOLDER, "*.pdf")
)

print("Total PDFs found:", len(pdf_files))

for pdf in pdf_files:
    print(os.path.basename(pdf))

Total PDFs found: 3
1706.03762v7.pdf
7051_understanding_how_encoder_deco (1).pdf
N19-1423 (1).pdf


In [4]:
for root, dirs, files in os.walk("/kaggle/input/datasets/priyankaadheli1/dataset"):
    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

/kaggle/input/datasets/priyankaadheli1/dataset/1706.03762v7.pdf
/kaggle/input/datasets/priyankaadheli1/dataset/7051_understanding_how_encoder_deco (1).pdf
/kaggle/input/datasets/priyankaadheli1/dataset/N19-1423 (1).pdf


In [5]:
all_documents = []

paper_info = {}

for pdf_path in pdf_files:

    # Open PDF using PyMuPDF
    pdf = fitz.open(pdf_path)

    pdf_metadata = pdf.metadata

    total_pages = len(pdf)

    filename = os.path.basename(pdf_path)

    paper_title = pdf_metadata.get("title")

    if not paper_title or paper_title.strip() == "":
        paper_title = os.path.splitext(filename)[0]

    # Generate unique paper ID
    paper_id = hashlib.md5(
        os.path.abspath(pdf_path).encode()
    ).hexdigest()[:8]

    # Store paper information
    paper_info[paper_id] = {
        "title": paper_title,
        "filename": filename,
        "path": pdf_path,
        "pages": total_pages
    }

    pdf.close()

    # Load PDF using Docling
    loader = DoclingLoader(file_path=pdf_path)

    documents = loader.load()

    # Attach paper metadata to documents
    for doc in documents:

        doc.metadata["paper_id"] = paper_id
        doc.metadata["paper_title"] = paper_title
        doc.metadata["filename"] = filename
        doc.metadata["total_pages"] = total_pages

    all_documents.extend(documents)

print("Total research papers loaded:", len(paper_info))
print("Total Docling documents:", len(all_documents))

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (4 outputs left out)

[WARNING] 2026-11-08 11:20:33,084 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:33,158 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:33,970 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:34,076 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:34,174 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in ava

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-11-08 11:20:56,266 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:56,346 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:57,550 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:57,670 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-08 11:20:57,809 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in ava

Total research papers loaded: 3
Total Docling documents: 230


In [6]:
def get_page_number(metadata):

    possible_keys = [
        "page",
        "page_number",
        "page_no",
        "page_index"
    ]

    for key in possible_keys:

        value = metadata.get(key)

        if value is not None:

            try:
                page_number = int(value)

                if key == "page_index":
                    page_number += 1

                return page_number

            except (ValueError, TypeError):
                pass

    # Check Docling provenance metadata
    provenance = metadata.get("dl_meta", {})

    if isinstance(provenance, str):
        try:
            provenance = json.loads(provenance)
        except (ValueError, TypeError):
            provenance = {}

    def search_page(data):

        if isinstance(data, dict):

            for key, value in data.items():

                if key in ["page_no", "page_number"]:

                    try:
                        return int(value)
                    except (ValueError, TypeError):
                        pass

                result = search_page(value)

                if result is not None:
                    return result

        elif isinstance(data, list):

            for item in data:

                result = search_page(item)

                if result is not None:
                    return result

        return None

    return search_page(provenance)

# Ingestion Summary

In [7]:
print("Ingestion Summary")

print("Number of papers:", len(paper_info))

total_pages = sum(
    info["pages"] for info in paper_info.values()
)

print("Total pages:", total_pages)

for paper_id, info in paper_info.items():

    paper_documents = [
        doc for doc in all_documents
        if doc.metadata["paper_id"] == paper_id
    ]

    print("\nPaper ID:", paper_id)

    print("Paper Title:", info["title"])

    print("Pages:", info["pages"])

    print("Docling documents:", len(paper_documents))

Ingestion Summary
Number of papers: 3
Total pages: 43

Paper ID: 36bad9a0
Paper Title: 1706.03762v7
Pages: 15
Docling documents: 67

Paper ID: fb1ecf1e
Paper Title: 7051_understanding_how_encoder_deco (1)
Pages: 12
Docling documents: 66

Paper ID: b0bee3ee
Paper Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Pages: 16
Docling documents: 97


In [8]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200,

    separators=[
        "\n\n",
        "\n",
        ".",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(all_documents)

print("Total chunks generated:", len(chunks))

Total chunks generated: 268


In [9]:
for index, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = str(index)

    page_number = get_page_number(chunk.metadata)

    if page_number is None:
        chunk.metadata["page_number"] = "Unknown"
    else:
        chunk.metadata["page_number"] = page_number

In [10]:
chunk_summary = []

for paper_id, info in paper_info.items():

    paper_chunks = [

        chunk for chunk in chunks

        if chunk.metadata["paper_id"] == paper_id

    ]

    chunk_summary.append({

        "Paper ID": paper_id,

        "Paper Title": info["title"],

        "Pages": info["pages"],

        "Chunks": len(paper_chunks)

    })

chunk_summary_df = pd.DataFrame(chunk_summary)

display(chunk_summary_df)

,Paper ID,Paper Title,Pages,Chunks
0,36bad9a0,1706.03762v7,15,76
1,fb1ecf1e,7051_understanding_how_encoder_deco (1),12,82
2,b0bee3ee,BERT: Pre-training of Deep Bidirectional Trans...,16,110


In [11]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-mpnet-base-v2"

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
CHROMA_PATH = "/kaggle/working/rag_db"

os.makedirs(CHROMA_PATH, exist_ok=True)

In [13]:
def create_chunk_id(chunk):

    text = chunk.page_content

    paper_id = chunk.metadata["paper_id"]

    page_no = chunk.metadata.get(
        "page_number",
        "Unknown"
    )

    unique_text = f"{paper_id}_{page_no}_{text}"

    return hashlib.md5(
        unique_text.encode()
    ).hexdigest()

In [14]:
for chunk in chunks:

    chunk.metadata["chunk_id"] = create_chunk_id(chunk)

print("Unique chunk IDs assigned successfully")

Unique chunk IDs assigned successfully


In [31]:
import json

# Clean complex metadata before storing in Chroma
for chunk in chunks:

    for key, value in list(chunk.metadata.items()):

        if isinstance(value, (dict, list, tuple)):

            chunk.metadata[key] = json.dumps(
                value,
                default=str
            )

        elif value is None:

            chunk.metadata.pop(key)


# Create Chroma vector store
COLLECTION_NAME = "research_papers"

vector_store = Chroma(

    collection_name=COLLECTION_NAME,

    embedding_function=embeddings,

    persist_directory=CHROMA_PATH

)


# Add chunks to vector database
vector_store.add_documents(

    documents=chunks,

    ids=[
        chunk.metadata["chunk_id"]
        for chunk in chunks
    ]

)

print("Documents stored in Chroma successfully")

print(
    "Total vectors stored:",
    vector_store._collection.count()
)

Documents stored in Chroma successfully
Total vectors stored: 268


In [33]:
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

In [34]:
def single_paper_retrieval(query, paper_id, k=4):

    results = vector_store.similarity_search(

        query,

        k=k,

        filter={"paper_id": paper_id}

    )

    return results

In [35]:
def cross_paper_retrieval(query, k_per_paper=2):

    all_results = []

    for paper_id in paper_info.keys():

        paper_results = vector_store.similarity_search(

            query,

            k=k_per_paper,

            filter={"paper_id": paper_id}

        )

        all_results.extend(paper_results)

    return all_results

In [36]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key loaded successfully")

Groq API key loaded successfully


In [37]:
llm = ChatGroq(

    model="openai/gpt-oss-120b",

    temperature=0

)

print("ChatGroq model loaded successfully")

ChatGroq model loaded successfully


In [38]:
prompt = ChatPromptTemplate.from_messages([

    ("system", """
    You are a research paper assistant.

    Answer the user's question using only the retrieved context.

    Rules:

    1. Do not use outside knowledge.
    2. Do not invent paper titles, page numbers,
       results, or citations.
    3. If the answer is not available in the context,
       clearly say that the information is not available.
    4. Compare research papers when multiple papers
       are provided.

    Retrieved Context:

    {context}
    """),

    ("human", "Question: {question}")

])

In [39]:
rag_chain = prompt | llm | StrOutputParser()

In [40]:
def build_context(documents):

    context = ""

    for index, doc in enumerate(documents):

        paper_title = doc.metadata.get(
            "paper_title",
            "Unknown Paper"
        )

        page_no = doc.metadata.get(
            "page_number",
            "Unknown"
        )

        context += f"""
        Chunk {index + 1}

        Paper: {paper_title}

        Page: {page_no}

        Content:
        {doc.page_content}

        """

    return context

In [41]:
def generate_answer(query, retrieved_docs):

    context = build_context(retrieved_docs)

    answer = rag_chain.invoke({

        "context": context,

        "question": query

    })

    return answer

In [42]:
def print_sources(retrieved_docs):

    print("\nSources:")

    seen_sources = set()

    for doc in retrieved_docs:

        paper_title = doc.metadata.get(
            "paper_title",
            "Unknown"
        )

        page_no = doc.metadata.get(
            "page_number",
            "Unknown"
        )

        source = (paper_title, page_no)

        if source not in seen_sources:

            seen_sources.add(source)

            print(
                f"Paper: {paper_title} | Page: {page_no}"
            )

In [43]:
def answer_question(query, mode="cross", paper_id=None):

    # Single-paper retrieval
    if mode == "single":

        if paper_id is None:
            raise ValueError("Please provide a valid Paper ID")

        if paper_id not in paper_info:
            raise ValueError("Invalid Paper ID")

        retrieved_docs = single_paper_retrieval(
            query,
            paper_id
        )

    # Cross-paper retrieval
    elif mode == "cross":

        retrieved_docs = cross_paper_retrieval(query)

    else:

        raise ValueError("Invalid retrieval mode")

    # Generate answer
    answer = generate_answer(
        query,
        retrieved_docs
    )

    print("\nResearch Assistant Answer:\n")

    print(answer)

    # Display sources
    print_sources(retrieved_docs)

    return answer

In [45]:
while True:

    print("\n------ Research Assistant ------")

    mode = input(
        "Choose mode (single/cross/exit): "
    ).strip().lower()

    if mode == "exit":

        print("Exiting Research Assistant")

        break

    if mode not in ["single", "cross"]:

        print("Invalid mode")

        continue

    query = input(
        "Enter your research question: "
    )

    if query.lower() == "exit":
        break

    paper_id = None

    if mode == "single":

        print("\nAvailable Research Papers:\n")

        for pid, info in paper_info.items():

            print(f"{pid}: {info['title']}")

        paper_id = input(
            "\nEnter Paper ID: "
        ).strip()

    answer_question(
        query,
        mode=mode,
        paper_id=paper_id
    )


------ Research Assistant ------


Choose mode (single/cross/exit):  cross
Enter your research question:  Compare the main objectives and findings of the research papers.



Research Assistant Answer:

**Paper 1 – “Attention Is All You Need” (1706.03762v7)**  
*Objective (as can be inferred from the title and author list)*:  To introduce a new neural architecture for sequence‑to‑sequence learning that relies solely on the attention mechanism, eliminating the need for recurrent or convolutional layers.  

*Findings*:  The retrieved excerpt contains only the abstract heading and the list of authors; no specific experimental results, performance numbers, or qualitative conclusions are provided in the context. Therefore the concrete findings of the paper are **not available** in the supplied material.

---

**Paper 2 – “BERT: Pre‑training of Deep Bidirectional Transformers for Language Understanding”**  
*Objective*:  To develop a method for pre‑training deep bidirectional Transformer models (BERT) that can be fine‑tuned on a variety of downstream language‑understanding tasks, and to evaluate the approach on standard benchmarks such as GLUE and SQuAD.  

*Fin

Choose mode (single/cross/exit):  single
Enter your research question:  Summarize this research paper



Available Research Papers:

36bad9a0: 1706.03762v7
fb1ecf1e: 7051_understanding_how_encoder_deco (1)
b0bee3ee: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding



Enter Paper ID:  b0bee3ee



Research Assistant Answer:

**Summary of “BERT: Pre‑training of Deep Bidirectional Transformers for Language Understanding” (based on the provided excerpts)**  

The paper introduces **BERT**, a deep bidirectional Transformer model that is first pre‑trained on large text corpora and then fine‑tuned for a variety of downstream language‑understanding tasks.  

### Key Experimental Setup  
- **GLUE Benchmark**: The authors evaluate BERT on the General Language Understanding Evaluation (GLUE) suite, which aggregates several sentence‑level classification and similarity tasks. The excerpts describe several of these tasks in detail:  

  - **SST‑2** – Binary sentiment classification on movie‑review sentences (Socher et al., 2013).  
  - **CoLA** – Binary classification of linguistic acceptability (Warstadt et al., 2018).  
  - **STS‑B** – Semantic textual similarity scoring (1–5) for sentence pairs from news headlines and other sources (Cer et al., 2017).  
  - **MRPC** – Paraphrase detectio

Choose mode (single/cross/exit):  exit


Exiting Research Assistant
